# Models — can we predict retail prices?

Predict `retail` from structured features, and test whether review-text blocks add
anything. Feature tables (`features_basic`, keywords, emb-PCA, anchors, full
embeddings) are precomputed in the pipeline and joined here on `wine_id`.

Flow: **baseline → target encoding → tuning → final CV**, each run with and without
`rating` (the strongest predictor).

## Methods to improve accuracy

**Done (≈ +0.05 R², 0.65 → 0.70):** target-encode the categoricals (fit on train),
tune + early-stop XGBoost, evaluate on a true held-out test set.

**Next:** hyper-parameter search (Optuna), LightGBM / CatBoost, log-target, a
quantile model for the trimmed long tail.

In [1]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

FEATURES_BASIC_PATH =   r"..\..\features\features_basic.parquet"
# KEYWORDS_PATH = r"..\..\features\features_keywords.parquet"
KEYWORDS_PATH =         r"..\..\features\features_keywords_robust.parquet"
EMBEDDINGS_PCA_PATH =   r"..\..\features\features_embeddings_PCA.parquet"
ANCHORED_PATH =         r"..\..\features\features_embeddings_anchored.parquet"
FULL_EMBEDDINGS_PATH =  r"..\..\features\features_embeddings.parquet"


## Load features

In [2]:
features = pd.read_parquet(FEATURES_BASIC_PATH)
keywords = pd.read_parquet(KEYWORDS_PATH)
embeddings_pca = pd.read_parquet(EMBEDDINGS_PCA_PATH)
embeddings_anchored = pd.read_parquet(ANCHORED_PATH)
full_embeddings = pd.read_parquet(FULL_EMBEDDINGS_PATH)
print(f"number of features in each data model:\n  features: {features.shape}  \n  keywords: {keywords.shape}  \n  "
      f"emb_pca: {embeddings_pca.shape}  \n  anchored: {embeddings_anchored.shape}  \n  full: {full_embeddings.shape}")

target = "retail"

basic_features = [
    "rating", "alcohol", "bottle_size", "vintage", "case_production",
    "country_ord", "wine_type_ord", "state_ord", "company_ord",
    "appellation_ord", "varietal_label_ord", "age_at_review",
]
basic_features_excl_rating = [f for f in basic_features if f != "rating"]

kw_features     = [c for c in keywords.columns if c.startswith("kw_") and not c.endswith("_count")]
pca_features    = [c for c in embeddings_pca.columns if c.startswith("pca_")]
anchor_features = [c for c in embeddings_anchored.columns if c.startswith("anchor_")]
full_features   = [c for c in full_embeddings.columns if c.startswith("emb_")]

df = (
    features
    .merge(keywords[["wine_id"] + kw_features], on="wine_id", how="left")
    .merge(embeddings_pca[["wine_id"] + pca_features], on="wine_id", how="left")
    .merge(embeddings_anchored[["wine_id"] + anchor_features], on="wine_id", how="left")
    .merge(full_embeddings[["wine_id"] + full_features], on="wine_id", how="left")
)
assert len(df) == len(features), "merge changed row count — wine_id not unique?"
print(f"Full table merged: {df.shape}")
print(f"basic features ({len(basic_features)}): {basic_features}")
print(f"aroma features ({len(kw_features)}): {kw_features}")
print(f"emb-PCA features ({len(pca_features)}): {pca_features}")
print(f"anchor features ({len(anchor_features)}): {anchor_features}")
print(f"full embeddings features ({len(full_features)}): {full_features}")


number of features in each data model:
  features: (135192, 15)  
  keywords: (135192, 107)  
  emb_pca: (135192, 21)  
  anchored: (135192, 23)  
  full: (135192, 385)
Full table merged: (135192, 494)
basic features (12): ['rating', 'alcohol', 'bottle_size', 'vintage', 'case_production', 'country_ord', 'wine_type_ord', 'state_ord', 'company_ord', 'appellation_ord', 'varietal_label_ord', 'age_at_review']
aroma features (53): ['kw_dry', 'kw_acidic', 'kw_tart', 'kw_sweet', 'kw_caramel', 'kw_alcohol', 'kw_strong', 'kw_balanced', 'kw_citrus', 'kw_apple', 'kw_pear', 'kw_strawberry', 'kw_raspberry', 'kw_cherry', 'kw_red', 'kw_black', 'kw_blackberry', 'kw_tropical', 'kw_banana', 'kw_pineapple', 'kw_lichi', 'kw_stone', 'kw_peach', 'kw_soil', 'kw_mineral', 'kw_oak', 'kw_coconut', 'kw_vanilla', 'kw_smoke', 'kw_tannic', 'kw_light', 'kw_heavy', 'kw_body', 'kw_flowers', 'kw_floral', 'kw_grass', 'kw_herbs', 'kw_spicy', 'kw_vegetables', 'kw_pepper', 'kw_bell', 'kw_earth', 'kw_leather', 'kw_tea', 'kw_

## Baseline — default XGB, simple split

Out-of-the-box `XGBRegressor()` on an 80/20 split, ordinal encoding, no tuning —
the reference the tuned pipeline below is measured against.

In [3]:
# Default out-of-the-box XGBoost with a plain 80/20 train-test split (no val set,
# no target encoding) — same simple recipe as 02_models_rating.

model_df = df[basic_features + kw_features + pca_features + anchor_features + full_features + [target]].dropna(subset=[target])
low, high = model_df[target].quantile([0.02, 0.90])
model_df = model_df[model_df[target].between(low, high)]
print(f"Kept retail ${low:.2f} - ${high:.2f}  ({len(model_df):,} rows)  | target = {target}")


# train and evaluate a model given a list of features — XGBRegressor with defaults made explicit
def train_eval_basic(feature_list):
    X, y = model_df[feature_list], model_df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = xgb.XGBRegressor(
        # core boosting
        n_estimators=100,
        learning_rate=0.3,
        max_depth=6,
        min_child_weight=1,
        gamma=0,
        # sampling
        subsample=1.0,
        colsample_bytree=1.0,
        # regularization
        reg_lambda=1.0,
        reg_alpha=0,
        # objective / method
        objective="reg:squarederror",
        booster="gbtree",
        tree_method="auto",
        # misc
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE":  mean_absolute_error(y_test, pred),
        "R2":   r2_score(y_test, pred),
    }



Kept retail $11.00 - $80.00  (113,220 rows)  | target = retail


### Compare feature blocks (with `rating`)

In [4]:
# default-XGB results across the same four feature blocks (with rating)
_, b1 = train_eval_basic(basic_features)
_, b2 = train_eval_basic(basic_features + kw_features)
_, b3 = train_eval_basic(basic_features + pca_features)
_, b4 = train_eval_basic(basic_features + anchor_features)
_, b5 = train_eval_basic(basic_features + full_features)

results_default = pd.DataFrame([
    {"model": "1: basic",            **b1},
    {"model": "2: basic + kw",       **b2},
    {"model": "3: basic + emb-PCA",  **b3},
    {"model": "4: basic + anchors",  **b4},
    {"model": "5: basic + full_embeddings",  **b5},
])
results_default["dR2_vs_base"] = results_default["R2"] - b1["R2"]
results_default.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: basic,12,10.4712,7.7238,0.6527,0.0000
1,2: basic + kw,65,10.7695,7.9598,0.6326,-0.0201
2,3: basic + emb-PCA,32,10.7885,8.0057,0.6313,-0.0214
3,4: basic + anchors,34,10.7822,7.9912,0.6317,-0.0209
4,5: basic + full_embeddings,396,11.2448,8.4237,0.5994,-0.0532


### Baseline — 5-fold CV

Variance check on the default-XGB baseline (train+val; the 15% test stays held out).

In [5]:
from sklearn.model_selection import KFold

train_df, tmp_df = train_test_split(model_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=42)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

cv_features = basic_features
cv_df = pd.concat([train_df, val_df])          # everything except the held-out test set
X_all, y_all = cv_df[cv_features], cv_df[target] # type: ignore

# Default XGBRegressor — only random_state / n_jobs fixed for reproducibility.
cv_params = dict(random_state=42, n_jobs=-1)

scores = []
for fold, (tr_idx, te_idx) in enumerate(KFold(5, shuffle=True, random_state=42).split(X_all), 1):
    X_tr, X_te = X_all.iloc[tr_idx], X_all.iloc[te_idx]
    y_tr, y_te = y_all.iloc[tr_idx], y_all.iloc[te_idx]
    m = xgb.XGBRegressor(**cv_params).fit(X_tr, y_tr)
    s = r2_score(y_te, m.predict(X_te))
    scores.append(s)
    print(f"fold {fold}: R2={s:.4f}")

scores = np.array(scores)
print(f"\n5-fold CV R2: {scores.mean():.4f} +/- {scores.std():.4f}")

train=79,254  val=16,983  test=16,983
fold 1: R2=0.6524
fold 2: R2=0.6585
fold 3: R2=0.6575
fold 4: R2=0.6498
fold 5: R2=0.6506

5-fold CV R2: 0.6538 +/- 0.0036


## Fine-tuning — target encoding + hyperparameters

70/15/15 train/val/test, shared across models so any difference comes from the
feature block. Two leakage-safe upgrades over the baseline: **target-encode** the
categoricals (fit on train) and **tune** the XGBoost params (early-stopped on val).
Metrics are on the held-out test set. The weak `basic + full_embeddings` block is
dropped from here on.

### Step 1 — target encoding only (default XGB)

In [6]:
# One model_df with every feature block, same rows for all models.
from sklearn.preprocessing import TargetEncoder

# Train / validation / test split (70 / 15 / 15). The 15% TEST set is held out
# and only used for the final metrics reported below; VALIDATION drives XGBoost
# early stopping (and would drive any hyper-parameter search). Same split for
# every model, so metric differences come from the feature block, not the rows.
train_df, tmp_df = train_test_split(model_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=42)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# The structured categoricals are ordinal-encoded with an arbitrary integer
# order (misleading for trees). Re-encode them with a CV-smoothed target mean,
# fit on TRAIN ONLY so no validation/test information leaks in.
CAT_ORD = ["country_ord", "wine_type_ord", "state_ord",
           "company_ord", "appellation_ord", "varietal_label_ord"]


def train_eval(feature_list):
    """Fit on train (early-stopped on val), report metrics on the held-out TEST set."""
    cats = [c for c in CAT_ORD if c in feature_list]
    X_train = train_df[feature_list].copy()
    X_val = val_df[feature_list].copy()
    X_test = test_df[feature_list].copy()
    y_train, y_val, y_test = train_df[target], val_df[target], test_df[target]

    if cats:  # leakage-safe target encoding: fit on train, apply to val/test
        enc = TargetEncoder(target_type="continuous", random_state=42)
        X_train[cats] = enc.fit_transform(train_df[cats], y_train)
        X_val[cats] = enc.transform(val_df[cats])
        X_test[cats] = enc.transform(test_df[cats])

    # model = make_model()
    model = xgb.XGBRegressor()

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = model.predict(X_test)
    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    }

train=79,254  val=16,983  test=16,983


In [7]:
# train and evaluate models with different feature sets, including rating
model1, m1 = train_eval(basic_features)
model2, m2 = train_eval(basic_features + kw_features)
model3, m3 = train_eval(basic_features + pca_features)
model4, m4 = train_eval(basic_features + anchor_features)

results = pd.DataFrame([
    {"model": "1: basic",            **m1},
    {"model": "2: basic + kw",       **m2},
    {"model": "3: basic + emb-PCA",  **m3},
    {"model": "4: basic + anchors",  **m4},
])
results["dR2_vs_base"] = results["R2"] - m1["R2"]
results.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: basic,12,9.9476,7.1870,0.6826,0.0000
1,2: basic + kw,65,10.0370,7.2977,0.6769,-0.0057
2,3: basic + emb-PCA,32,10.0906,7.3544,0.6734,-0.0092
3,4: basic + anchors,34,10.0905,7.3371,0.6734,-0.0092


### Step 2 — target encoding + tuned XGB

In [8]:
# One model_df with every feature block, same rows for all models.
from sklearn.preprocessing import TargetEncoder

# Train / validation / test split (70 / 15 / 15). The 15% TEST set is held out
# and only used for the final metrics reported below; VALIDATION drives XGBoost
# early stopping (and would drive any hyper-parameter search). Same split for
# every model, so metric differences come from the feature block, not the rows.
train_df, tmp_df = train_test_split(model_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=42)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# The structured categoricals are ordinal-encoded with an arbitrary integer
# order (misleading for trees). Re-encode them with a CV-smoothed target mean,
# fit on TRAIN ONLY so no validation/test information leaks in.
CAT_ORD = ["country_ord", "wine_type_ord", "state_ord",
           "company_ord", "appellation_ord", "varietal_label_ord"]


# Change the XGBRegressor parameters - at hand tuning.
def make_model():
    """Tuned, early-stopped XGBoost (replaces the previous default XGBRegressor())."""
    return xgb.XGBRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=2.0,
        early_stopping_rounds=80,
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
    )


def train_eval(feature_list, params=None):
    """Fit on train (early-stopped on val), report metrics on the held-out TEST set.

    `params=None` -> use make_model() (early-stopped). Pass a params dict (e.g.
    `cv_params`, which has no early_stopping_rounds) to train that config instead.
    """
    cats = [c for c in CAT_ORD if c in feature_list]
    X_train = train_df[feature_list].copy()
    X_val = val_df[feature_list].copy()
    X_test = test_df[feature_list].copy()
    y_train, y_val, y_test = train_df[target], val_df[target], test_df[target]

    if cats:  # leakage-safe target encoding: fit on train, apply to val/test
        enc = TargetEncoder(target_type="continuous", random_state=42)
        X_train[cats] = enc.fit_transform(train_df[cats], y_train)
        X_val[cats] = enc.transform(val_df[cats])
        X_test[cats] = enc.transform(test_df[cats])

    model = make_model() if params is None else xgb.XGBRegressor(**params)

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = model.predict(X_test)
    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    }

train=79,254  val=16,983  test=16,983


In [9]:
# train and evaluate models with different feature sets, including rating
model1, m1 = train_eval(basic_features)
model2, m2 = train_eval(basic_features + kw_features)
model3, m3 = train_eval(basic_features + pca_features)
model4, m4 = train_eval(basic_features + anchor_features)

results = pd.DataFrame([
    {"model": "1: basic",            **m1},
    {"model": "2: basic + kw",       **m2},
    {"model": "3: basic + emb-PCA",  **m3},
    {"model": "4: basic + anchors",  **m4},
])
results["dR2_vs_base"] = results["R2"] - m1["R2"]
results.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: basic,12,9.6273,6.9358,0.7027,0.0000
1,2: basic + kw,65,9.7252,7.0417,0.6966,-0.0061
2,3: basic + emb-PCA,32,9.7570,7.0822,0.6947,-0.0081
3,4: basic + anchors,34,9.7905,7.0991,0.6926,-0.0102


### 5-fold CV — final config

Variance estimate for the final model (target encoding + tuned XGB) on train+val;
encoding re-fit inside each fold. Test set untouched.

In [10]:
from sklearn.model_selection import KFold

cv_features = basic_features
cv_df = pd.concat([train_df, val_df])          # train + val; the 15% test set stays untouched
X_all, y_all = cv_df[cv_features], cv_df[target]
cats = [c for c in CAT_ORD if c in cv_features]

# Final tuned config. There is no early-stopping fold inside CV, so n_estimators
# is fixed near the early-stopped optimum found above.
cv_params = dict(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=2.0,
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

# 5-fold CV: each fold trains on 4/5 of the rows and is scored on the held-out 1/5.
# Target encoding is re-fit inside every fold (on that fold's training rows only),
# so no information leaks from the held-out rows.
fold_scores = []
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_num, (train_idx, holdout_idx) in enumerate(kfold.split(X_all), start=1):
    # this fold's training rows vs its held-out (scored) rows
    X_train_fold, X_holdout_fold = X_all.iloc[train_idx].copy(), X_all.iloc[holdout_idx].copy()
    y_train_fold, y_holdout_fold = y_all.iloc[train_idx], y_all.iloc[holdout_idx]

    # leakage-safe target encoding: fit on the training rows, apply to both
    encoder = TargetEncoder(target_type="continuous", random_state=42)
    X_train_fold[cats] = encoder.fit_transform(X_train_fold[cats], y_train_fold)
    X_holdout_fold[cats] = encoder.transform(X_holdout_fold[cats])

    # train on the training rows, score R² on the held-out rows
    model_fold = xgb.XGBRegressor(**cv_params).fit(X_train_fold, y_train_fold)
    fold_r2 = r2_score(y_holdout_fold, model_fold.predict(X_holdout_fold))
    fold_scores.append(fold_r2)
    print(f"fold {fold_num}: R2={fold_r2:.4f}")

fold_scores = np.array(fold_scores)
print(f"\n5-fold CV R2: {fold_scores.mean():.4f} +/- {fold_scores.std():.4f}")

fold 1: R2=0.7052
fold 2: R2=0.7172
fold 3: R2=0.7136
fold 4: R2=0.7050
fold 5: R2=0.7057

5-fold CV R2: 0.7093 +/- 0.0051


# Retail prediction without `rating`

`rating` is the strongest predictor — drop it to see how much price leans on the
score, and whether any text block recovers the lost signal. Mirrors the steps above.

### Baseline — default XGB

In [11]:
# Same models, but with `rating` dropped from the feature set.
# rating is the strongest structured predictor of price, so this shows how much
# price prediction leans on the score — and whether the text blocks recover any
# of the lost signal when rating is unavailable. (Models 5-8 mirror 1-4.)
model1, m1 = train_eval_basic(basic_features_excl_rating)
model2, m2 = train_eval_basic(basic_features_excl_rating + kw_features)
model3, m3 = train_eval_basic(basic_features_excl_rating + pca_features)
model4, m4 = train_eval_basic(basic_features_excl_rating + anchor_features)

results_excl = pd.DataFrame([
    {"model": "1: no-rating base",       **m1},
    {"model": "2: no-rating + kw",       **m2},
    {"model": "3: no-rating + emb-PCA",  **m3},
    {"model": "4: no-rating + anchors",  **m4},
])
results_excl["dR2_vs_base"] = results_excl["R2"] - m1["R2"]
results_excl.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,1: no-rating base,11,11.2638,8.3669,0.5981,0.0000
1,2: no-rating + kw,64,11.5494,8.6562,0.5774,-0.0206
2,3: no-rating + emb-PCA,31,11.7273,8.7903,0.5643,-0.0338
3,4: no-rating + anchors,33,11.7266,8.7693,0.5644,-0.0337


### Target encoding + tuned XGB

In [12]:
# Same models, but with `rating` dropped from the feature set.
# rating is the strongest structured predictor of price, so this shows how much
# price prediction leans on the score — and whether the text blocks recover any
# of the lost signal when rating is unavailable. (Models 5-8 mirror 1-4.)
model5, m5 = train_eval(basic_features_excl_rating)
model6, m6 = train_eval(basic_features_excl_rating + kw_features)
model7, m7 = train_eval(basic_features_excl_rating + pca_features)
model8, m8 = train_eval(basic_features_excl_rating + anchor_features)

results_excl = pd.DataFrame([
    {"model": "5: no-rating base",       **m5},
    {"model": "6: no-rating + kw",       **m6},
    {"model": "7: no-rating + emb-PCA",  **m7},
    {"model": "8: no-rating + anchors",  **m8},
])
results_excl["dR2_vs_base"] = results_excl["R2"] - m5["R2"]
results_excl.round(4)

,model,n_features,RMSE,MAE,R2,dR2_vs_base
0,5: no-rating base,11,10.2764,7.4452,0.6613,0.0000
1,6: no-rating + kw,64,10.2822,7.4825,0.6609,-0.0004
2,7: no-rating + emb-PCA,31,10.3802,7.5718,0.6544,-0.0069
3,8: no-rating + anchors,33,10.4302,7.5988,0.6511,-0.0102


# Feature Importance

## Feature importance — final model (with `rating`)

Final config (tuned XGB, no early stopping via `params=cv_params`), test R² reported.

In [13]:
# Final model via train_eval, but with cv_params instead of make_model() — i.e.
# the tuned config WITHOUT early stopping (cv_params has no early_stopping_rounds).
# train_eval fits on train, target-encodes, and reports held-out TEST metrics.
final_model, final_metrics = train_eval(basic_features, params=cv_params)
print(f"Final model (with rating) — test R2={final_metrics['R2']:.4f}  "
      f"RMSE={final_metrics['RMSE']:.3f}  MAE={final_metrics['MAE']:.3f}")

imp = (
    pd.DataFrame({"feature": basic_features, "importance": final_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .assign(importance=lambda d: d["importance"].round(3))
)
show(imp)

Final model (with rating) — test R2=0.7027  RMSE=9.627  MAE=6.936


Loading ITables v2.7.3 from the internet... (need help?)


## Feature importance — final model (without `rating`)

Same config, `rating` dropped.

In [14]:
# Same as the with-rating cell, but WITHOUT `rating`: train_eval with cv_params
# (tuned config, no early stopping). Fits on train, reports held-out TEST metrics.
final_model_nr, final_metrics_nr = train_eval(basic_features_excl_rating, params=cv_params)
print(f"Final model (no rating) — test R2={final_metrics_nr['R2']:.4f}  "
      f"RMSE={final_metrics_nr['RMSE']:.3f}  MAE={final_metrics_nr['MAE']:.3f}")

imp = (
    pd.DataFrame({"feature": basic_features_excl_rating, "importance": final_model_nr.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .assign(importance=lambda d: d["importance"].round(3))
)
show(imp)

Final model (no rating) — test R2=0.6613  RMSE=10.276  MAE=7.445


Loading ITables v2.7.3 from the internet... (need help?)


# SHAP — signed dollar effect (the "slope" analog)

Tree-SHAP splits each prediction additively, **in dollars**. Per feature:
`mean_abs_$` = average contribution size (importance in `signed_$`) that
magnitude signed by direction (**+** raises price, **−** lowers it). Exact tree SHAP
via XGBoost's `pred_contribs` (no extra library), on a test-set sample.

In [15]:
def shap_signed_dollars(model, feature_list, explain_df, n_sample=4000):
    """Per-feature SHAP summary in dollars: mean |contribution| and its signed direction.

    Uses XGBoost's exact tree SHAP (`pred_contribs`). Categoricals are re-encoded
    with the same train-fitted TargetEncoder the model was trained on.
    """
    cats = [c for c in CAT_ORD if c in feature_list]
    sample = explain_df.sample(min(n_sample, len(explain_df)), random_state=42)
    X = sample[feature_list].copy()
    if cats:
        enc = TargetEncoder(target_type="continuous", random_state=42)
        enc.fit(train_df[cats], train_df[target])
        X[cats] = enc.transform(sample[cats])

    contribs = model.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)[:, :-1]  # drop bias col
    mean_abs = np.abs(contribs).mean(axis=0)
    # direction: sign of feature <-> contribution correlation (pandas .corr drops NaN pairs)
    sign = [np.sign(X[f].reset_index(drop=True).corr(pd.Series(contribs[:, j])))
            for j, f in enumerate(feature_list)]
    return (
        pd.DataFrame({"feature": feature_list,
                      "mean_abs_$": mean_abs.round(2),
                      "signed_$": (np.array(sign) * mean_abs).round(2)})
        .sort_values("mean_abs_$", ascending=False)
        .reset_index(drop=True)
    )

In [16]:
print("Final model WITH rating — signed SHAP effect on price ($):")
show(shap_signed_dollars(final_model, basic_features, test_df))

Final model WITH rating — signed SHAP effect on price ($):


Loading ITables v2.7.3 from the internet... (need help?)


In [17]:
print("Final model WITHOUT rating — signed SHAP effect on price ($):")
show(shap_signed_dollars(final_model_nr, basic_features_excl_rating, test_df))

Final model WITHOUT rating — signed SHAP effect on price ($):


Loading ITables v2.7.3 from the internet... (need help?)


## SHAP $ effect vs gain importance — scatter (with `rating`)

Two views of *what drives price* on the same final model: XGBoost **gain
importance** (x) and **mean |SHAP|** in dollars (y), one point per feature. Points
on a rising diagonal mean the two rankings agree; the SHAP axis adds a real dollar
scale, and colour shows whether the feature pushes price **up** or **down**
(`signed_$`). Spearman ρ in the title quantifies the rank agreement.

In [18]:
# Scatter: gain feature importance (x) vs mean |SHAP| in $ (y), one point per
# feature, for the final WITH-rating model. Both rank what drives price; this shows
# they agree (Spearman rho) and adds the SHAP dollar scale + up/down direction.
import plotly.express as px

shap_df = shap_signed_dollars(final_model, basic_features, test_df)
imp_df = pd.DataFrame({"feature": basic_features,
                       "importance": final_model.feature_importances_})

plot_df = (
    shap_df.merge(imp_df, on="feature")
    .assign(
        label=lambda d: d["feature"].str.replace("_ord", "", regex=False),
        direction=lambda d: np.where(d["signed_$"] >= 0, "raises price (+)", "lowers price (−)"),
    )
)

rho = plot_df["importance"].corr(plot_df["mean_abs_$"], method="spearman")

fig = px.scatter(
    plot_df, x="importance", y="mean_abs_$",
    text="label", color="direction",
    color_discrete_map={"raises price (+)": "#2c7fb8", "lowers price (−)": "#d95f0e"},
    hover_data={"signed_$": ":.2f", "importance": ":.3f", "mean_abs_$": ":.2f", "label": False},
    title=f"Retail model (with rating): SHAP $ effect vs gain importance "
          f"&mdash; Spearman ρ = {rho:.2f}",
    labels={"importance": "Gain feature importance",
            "mean_abs_$": "Mean |SHAP|  ($ per prediction)",
            "direction": "SHAP direction"},
)
fig.update_traces(textposition="top center",
                  marker=dict(size=13, line=dict(width=1, color="white")))
fig.update_layout(height=600, width=950)
fig.show()

## Per-country SHAP — how each country shifts predicted price

The feature-level SHAP above lumps all of `country` into one number. Here we open it
up **per category value**: for every test row we take the SHAP contribution of the
`country` feature (in dollars), then average it within each country. The bar is the
mean `$` that *being from that country* moves the model's predicted retail, up or
down, holding the rest of the wine's attributes fixed. Ordinal codes are mapped back
to country names via Silver, and countries with `< 30` test rows are dropped so the
averages aren't noise.

In [19]:
# Per-category SHAP for `country`: average the country feature's $ contribution
# within each country, on the held-out test set (final WITH-rating model).
SILVER_PATH = r"..\..\.data\wine_reviews_silver.parquet"
MIN_ROWS = 30  # drop countries with too few test rows to average reliably

# country_ord -> country name (bijective ordinal encoding), rebuilt via Silver.
silver = pd.read_parquet(SILVER_PATH, columns=["wine_id", "country"])
ord2country = (
    features[["wine_id", "country_ord"]]
    .merge(silver, on="wine_id")
    .dropna(subset=["country_ord"])
    .drop_duplicates("country_ord")
    .set_index("country_ord")["country"]
)

# Re-encode the categoricals exactly as the model was trained (TargetEncoder fit on
# train), then take the per-row SHAP contribution of the `country` column.
enc = TargetEncoder(target_type="continuous", random_state=42)
enc.fit(train_df[CAT_ORD], train_df[target])
X_test_enc = test_df[basic_features].copy()
X_test_enc[CAT_ORD] = enc.transform(test_df[CAT_ORD])

contribs = final_model.get_booster().predict(xgb.DMatrix(X_test_enc), pred_contribs=True)[:, :-1]
country_shap = contribs[:, basic_features.index("country_ord")]

by_country = (
    pd.DataFrame({"country_ord": test_df["country_ord"].to_numpy(), "shap_$": country_shap})
    .groupby("country_ord")
    .agg(mean_shap=("shap_$", "mean"), n=("shap_$", "size"))
    .reset_index()
    .assign(country=lambda d: d["country_ord"].map(ord2country))
    .query("n >= @MIN_ROWS")
    .sort_values("mean_shap")
)
by_country["direction"] = np.where(by_country["mean_shap"] >= 0,
                                   "raises price (+)", "lowers price (−)")

fig = px.bar(
    by_country, x="mean_shap", y="country", orientation="h",
    color="direction",
    color_discrete_map={"raises price (+)": "#2c7fb8", "lowers price (−)": "#d95f0e"},
    hover_data={"n": True, "mean_shap": ":.2f", "country": False, "direction": False},
    title="Per-country SHAP — mean $ effect on predicted retail (with-rating model, test set)",
    labels={"mean_shap": "Mean SHAP contribution  ($ vs. average wine)",
            "country": "", "direction": "SHAP direction"},
)
fig.add_vline(x=0, line_width=1, line_color="gray")
fig.update_layout(height=650, width=900)
fig.show()

# Conclusions

## Feature-block A/B — baseline model

Default XGB (ordinal encoding, 80/20 split), `rating` included — which dataset
wins **before** any tuning:

| feature block | test R² |
|---|---|
| **basic** | **0.653** |
| basic + keywords | 0.633 |
| basic + emb-PCA | 0.631 |
| basic + anchors | 0.632 |
| basic + full embeddings | 0.599 |

No text block beats plain `basic` — review text doesn't help price even at
baseline. `basic` is carried forward.

## Feature-block A/B — final model (target encoding + tuned XGB)

Held-out test R² per block, with and without `rating`:

| feature block | with `rating` | without `rating` |
|---|---|---|
| **basic** | **0.703** | **0.661** |
| basic + keywords | 0.697 | 0.661 |
| basic + emb-PCA | 0.695 | 0.654 |
| basic + anchors | 0.693 | 0.651 |

Tuning lifts every block but the ranking holds — text still doesn't help, and no
block recovers the signal lost when `rating` is dropped.

## Model development — stage by stage

Best block (`basic`), held-out test R², with and without `rating`. A reviewer
score is usually **unavailable** when pricing a wine, so the *without-`rating`*
column is the realistic deployment case:

| stage | with `rating` | without `rating` |
|---|---|---|
| Baseline — default XGB, ordinal encoding | 0.653 | 0.598 |
| + Target encoding | 0.683 | — |
| + Target encoding + hyperparameter tuning | 0.703 | 0.661 |
| **5-fold CV** (final, with `rating`) | **0.709 ± 0.005** | — |

Net **+0.05 R²** with `rating` (0.653 → 0.703) and **+0.06** without it
(0.598 → 0.661) — no new features, no leakage. RMSE 10.47 → 9.63, MAE 7.72 → 6.94.

## With vs without `rating`

`rating` is the single strongest predictor, but normally unavailable at pricing
time, so **0.661** is what matters in practice. Dropping it costs ~0.04 R², which
region and producer largely absorb (see below) — price stays ~0.66 predictable
from structured attributes alone.

## What drives price — feature importance + SHAP

Gain importance and SHAP agree on the ranking. Top drivers (final model):  
`appellation` (imp 0.30) > `rating` (0.20) > `age_at_review` (0.13) > `company` (0.10).

SHAP signed dollar effect — average $ a feature moves the prediction, with sign:

| feature | signed $ |
|---|---|
| appellation | +5.4 |
| rating | +3.7 |
| company | +3.5 |
| case_production | −2.6 |
| age_at_review | +2.6 |
| alcohol | +1.2 |

Reading the signs:  
- region, score and producer push price **up** — premium appellations and  
  reputable wineries command more;  
- **higher production volume pushes price down** (−$2.6) — mass-produced = cheaper;  
- older bottles and higher alcohol nudge price up.  

Without `rating`, `appellation` (+$5.8) and `company` (+$4.0) grow to fill the gap —  
the model leans harder on *where* and *who*.

## Text features

Review-text blocks (kw / emb-PCA / anchors) help neither model — structured  
features carry price; prose tracks `rating` (see `02_models_rating`).